# Winter-season episode evaluation (offline)

Reproducible workflow on the **bundled synthetic** Delhi–NCT winter fixture.
No live CPCB/OpenAQ/FIRMS calls. The series is a method-development construct, **not** observational ground truth.

Requires an editable install from the repo root: `pip install -e .` (optional: `pip install -e ".[eval]"` for Jupyter/matplotlib).

In [ ]:
from __future__ import annotations

from aether_agents.eval import (
    detect_episodes,
    demo_scorecards,
    episode_detection_metrics,
    load_synthetic_winter,
    missingness_report,
    rubric_catalog,
)

data = load_synthetic_winter()
meta = data["meta"]
obs = data["observed_aqi"]
pred = data["predicted_aqi"]
print(meta["name"], meta["season_label"], f"n_hours={meta['n_hours']}")
print(meta["disclaimer"])

## Missing-data diagnostics

Gaps stay gaps. Coverage and gap-run statistics are reported before any pairwise score.

In [ ]:
print("observed:", missingness_report(obs))
print("predicted:", missingness_report(pred))

## Episode detection on the reference series

In [ ]:
episodes = detect_episodes(obs, min_duration_hours=6)
print(f"detected episodes: {len(episodes)}")
for ep in episodes[:8]:
    print(
        f"  hours [{ep.start}, {ep.end}) peak@{ep.peak_index} "
        f"aqi={ep.peak_aqi:.0f} severity={ep.severity}"
    )
if len(episodes) > 8:
    print(f"  ... {len(episodes) - 8} more")

## Pairwise detection metrics

Only hours where **both** series are non-missing contribute to precision / recall / F1.

In [ ]:
metrics = episode_detection_metrics(obs, pred, min_duration_hours=6)
for key, value in metrics.to_dict().items():
    print(f"{key}: {value}")

## Optional quicklook plot

Skipped automatically if matplotlib is not installed.

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    print("matplotlib not installed; skipping plot (pip install -e \".[eval]\")")
else:
    # Plot a 14-day slice covering the synthetic stubble → Diwali region.
    start, end = 14 * 24, 48 * 24
    xs = list(range(start, end))
    yo = [obs[i] for i in xs]
    yp = [pred[i] for i in xs]
    fig, ax = plt.subplots(figsize=(10, 3.5))
    ax.plot(xs, yo, label="observed (synthetic)", lw=1.2)
    ax.plot(xs, yp, label="predicted (synthetic)", lw=1.0, alpha=0.85)
    ax.axhline(301, color="0.4", ls="--", lw=0.8, label="Very Poor (≥301)")
    ax.set_xlabel("hour index")
    ax.set_ylabel("AQI")
    ax.set_title("Synthetic winter slice (stubble / Diwali region)")
    ax.legend(loc="upper right", fontsize=8)
    fig.tight_layout()
    plt.show()

## Case-study rubrics

Diwali, post-monsoon stubble/transport, and local stagnation. Scores below are **demo** scorecards for notebook reproducibility; they are not field audits and do not claim measurement ground truth.

In [ ]:
catalog = rubric_catalog()
print(catalog["disclaimer"])
print()
for case_id, info in catalog["cases"].items():
    print(f"## {case_id}: {info['label']}")
    print(info["narrative"])
    print("criteria:", ", ".join(c["criterion_id"] for c in info["criteria"]))
    print()

for card in demo_scorecards():
    d = card.to_dict()
    print(
        f"{d['case_id']}: {d['total']}/{d['max_total']} "
        f"(normalized={d['normalized']:.2f}): {d['overall_notes']}"
    )

## Takeaways

1. Episode membership is scored only on jointly observed hours; missingness is reported explicitly.
2. Case rubrics support transparent literature-aligned review for three winter archetypes.
3. Swap the synthetic fixture for holdout observational winters later without changing the metric APIs; keep the same gap rules.

See `docs/methods.md` for the poster/paper methods note.